In [1]:
STORE         = "/kaggle/input/datasets/tataruteodor/crop-store-m020"           # <- adjust
STORE_OVERLAY = "/kaggle/input/datasets/tataruteodor/crop-store-m020-overlay"   # <- adjust
SRC_DATASET   = "/kaggle/input/datasets/tataruteodor/src-ablation"                       # <- adjust (contains src/)
OUT = "/kaggle/working/ablations_kg"
EPOCHS = 200          # local cap (config.MAX_EPOCHS); raise here if the chat asks for a higher cap

# (model, ablate, lr, store, tag_suffix) - order is priority order.
RUNS = [
    ("regularised", None,      None, "plain",   "_m020"),           # baseline replica
    ("regularised", "dropout", None, "plain",   "_m020"),           # A4.1
    ("regularised", "l2",      None, "plain",   "_m020"),           # A4.2
    ("regularised", "se",      None, "plain",   "_m020"),           # A4.3
    ("regularised", "aug",     None, "plain",   "_m020"),           # A4.4
    ("regularised", None,      2e-3, "plain",   "_m020"),           # A4.5
    ("regularised", None,      None, "overlay", "_m020_overlay_lrmatched"),  # A4.6
    ("vgg16",       None,      None, "overlay", "_m020_overlay"),   # A4.7 (transfer path: see cell 3)
]

In [2]:
import os, shutil, sys, json, time
import tensorflow as tf

if os.path.exists("/kaggle/working/src"):
    shutil.rmtree("/kaggle/working/src")
shutil.copytree(os.path.join(SRC_DATASET, "src"), "/kaggle/working/src")
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")
print("TF", tf.__version__, "| GPUs", tf.config.list_physical_devices("GPU"))
tf.keras.mixed_precision.set_global_policy("float32")

from src import config, train_crop_store
for d in (STORE, STORE_OVERLAY):
    print(d, sorted(os.listdir(d)))
    print(json.load(open(os.path.join(d, "manifest.json")))["splits"])

TF 2.20.0 | GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
/kaggle/input/datasets/tataruteodor/crop-store-m020 ['manifest.json', 'train_images.npy', 'train_meta.csv', 'val_images.npy', 'val_meta.csv']
{'val': {'n': 243, 'n_malignant': 112, 'shape': [224, 224, 1], 'mean': 0.49961215257644653, 'std': 0.26650530099868774}, 'train': {'n': 1210, 'n_malignant': 559, 'shape': [224, 224, 1], 'mean': 0.5095047950744629, 'std': 0.27011674642562866}}
/kaggle/input/datasets/tataruteodor/crop-store-m020-overlay ['manifest.json', 'train_images.npy', 'train_meta.csv', 'val_images.npy', 'val_meta.csv']
{'val': {'n': 243, 'n_malignant': 112, 'shape': [224, 224, 1], 'mean': 0.39507535099983215, 'std': 0.32987645268440247}, 'train': {'n': 1210, 'n_malignant': 559, 'shape': [224, 224, 1], 'mean': 0.40019676089286804, 'std': 0.33346328139305115}}


I0000 00:00:1788862907.904686      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [3]:
results = []
for model, ablate, lr, store, suffix in RUNS:
    if model != "regularised":
        continue                       # the VGG16 run is cell 3
    s = STORE if store == "plain" else STORE_OVERLAY
    t0 = time.time()
    r = train_crop_store.main(s, model=model, ablate=ablate, lr=lr, epochs=EPOCHS,
                              tag_suffix=suffix, platform_suffix="_kg", out_dir=OUT)
    results.append(r)
    print(f"### {r['tag']}: val_auc {r['val_auc_best']:.4f} @ {r['best_epoch']}/{r['epochs_run']} "
          f"in {time.time() - t0:.0f} s", flush=True)
json.dump(results, open(f"{OUT}/summary.json", "w"), indent=2)
for r in results:
    print(f"{r['tag']:<48} {r['val_auc_best']:.4f}  epoch {r['best_epoch']}/{r['epochs_run']}")

[data_loader] Class counts (train): {0: 651, 1: 559}
[data_loader] Class weights:        {0: 0.9293394777265745, 1: 1.0822898032200359}
[train_crop_store] regularised_crop_m020_kg: lr=0.0005 augment=True ablate=None store=crop-store-m020 rows=1210 epochs=200
Epoch 1/200


I0000 00:00:1788862938.794966      67 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



Epoch 1: val_auc improved from None to 0.49669, saving model to /kaggle/working/ablations_kg/regularised_crop_m020_kg_best.weights.h5

Epoch 1: finished saving model to /kaggle/working/ablations_kg/regularised_crop_m020_kg_best.weights.h5
38/38 - 46s - 1s/step - accuracy: 0.5380 - auc: 0.5418 - loss: 0.8647 - val_accuracy: 0.4691 - val_auc: 0.4967 - val_loss: 0.7659 - learning_rate: 5.0000e-04
Epoch 2/200

Epoch 2: val_auc improved from 0.49669 to 0.51540, saving model to /kaggle/working/ablations_kg/regularised_crop_m020_kg_best.weights.h5

Epoch 2: finished saving model to /kaggle/working/ablations_kg/regularised_crop_m020_kg_best.weights.h5
38/38 - 4s - 104ms/step - accuracy: 0.5256 - auc: 0.5509 - loss: 0.8294 - val_accuracy: 0.4733 - val_auc: 0.5154 - val_loss: 0.7780 - learning_rate: 5.0000e-04
Epoch 3/200

Epoch 3: val_auc improved from 0.51540 to 0.53142, saving model to /kaggle/working/ablations_kg/regularised_crop_m020_kg_best.weights.h5

Epoch 3: finished saving model to /k

In [4]:
# train_crop_store trains single-stage models. The two-stage VGG16 run keeps
# the local path (gpu_queue_a4.sh overlay), ~1 h on the laptop, unless a
# two-stage store trainer is added later. Left here as the marker.
print("A4.7 stays local: bash notebooks/scripts/gpu_queue_a4.sh overlay (runs only the VGG16 line once the regularised overlay _kg result exists)")

A4.7 stays local: bash notebooks/scripts/gpu_queue_a4.sh overlay (runs only the VGG16 line once the regularised overlay _kg result exists)


In [5]:
# Download convention on the laptop: unzip into outputs/ so that
#   outputs/results/{tag}_history.json, outputs/weights/{tag}_best.weights.h5,
#   outputs/logs/{tag}_training.csv land beside the local runs (tags carry _kg).
print(sorted(os.listdir(OUT)))
shutil.make_archive("/kaggle/working/ablations_kg", "zip", OUT)
print("-> /kaggle/working/ablations_kg.zip")

['regularised_crop_m020_kg_best.weights.h5', 'regularised_crop_m020_kg_final.weights.h5', 'regularised_crop_m020_kg_history.json', 'regularised_crop_m020_kg_manifest.json', 'regularised_crop_m020_kg_training.csv', 'regularised_crop_m020_lr2e-3_kg_best.weights.h5', 'regularised_crop_m020_lr2e-3_kg_final.weights.h5', 'regularised_crop_m020_lr2e-3_kg_history.json', 'regularised_crop_m020_lr2e-3_kg_manifest.json', 'regularised_crop_m020_lr2e-3_kg_training.csv', 'regularised_crop_m020_no_aug_kg_best.weights.h5', 'regularised_crop_m020_no_aug_kg_final.weights.h5', 'regularised_crop_m020_no_aug_kg_history.json', 'regularised_crop_m020_no_aug_kg_manifest.json', 'regularised_crop_m020_no_aug_kg_training.csv', 'regularised_crop_m020_no_dropout_kg_best.weights.h5', 'regularised_crop_m020_no_dropout_kg_final.weights.h5', 'regularised_crop_m020_no_dropout_kg_history.json', 'regularised_crop_m020_no_dropout_kg_manifest.json', 'regularised_crop_m020_no_dropout_kg_training.csv', 'regularised_crop_m020